# Homenwork 2: Discovery of Frequent Itemsets and Association Rules

## Task 1: Find frequent itemsets with support at least s

We define a minimal $s$=0.01 and start the first pass: load all "buckets" into memory and counts the appearance of each single item (singletons). Then filter out the items that has support bigger than defined $s$. 

We start our second pass based on the frequent singletons. First, we generate all possible pairs according to combinations of frequent singletons. This early pruning is based on the observation (monotocity of support) that if a set is not a frequent set then its superset is not a frequent set. Then we filter the doubletons that truly has support larger than $s$ 

Similarly, we continue our third and fourth and k-th pass to find frequent set of k items. The process stops when no frequent itemset is generated at k-th pass.

In [25]:
import collections
from itertools import combinations
# First pass: load data and count singletons
def load_and_count_singletons(file_path,s_percent):
    item_counts = collections.defaultdict(int)
    total_baskets = 0
    singleton_sets = set()
    with open(file_path, 'r') as f:
        for line in f:
            total_baskets += 1
            items = line.strip().split(' ')
            for item in items:
                item_counts[item] += 1
        support = int(s_percent * total_baskets)

        # Filter singletons
        frequent_singletons = {item for item, count in item_counts.items() if count >= support}
        return frequent_singletons, support, total_baskets, item_counts
# Second pass: generate candidate pairs and filter frequent pairs
def get_frequent_pairs(file_path, frequent_singletons, support):
    candidate_pairs = set(combinations(sorted(frequent_singletons), 2))  # 排序
    pair_counts = collections.defaultdict(int)

    with open(file_path, 'r') as f:
        for line in f:
            items = line.strip().split(' ')
            # Only consider items that are frequent singletons (Prune step)
            basket_items = [item for item in items if item in frequent_singletons]
            
            basket_pairs = combinations(sorted(basket_items), 2)  # 排序
            for pair in basket_pairs:
                if pair in candidate_pairs:
                    pair_counts[pair] += 1
    
    frequent_pairs = {pair: count for pair, count in pair_counts.items() if count >= support}
    return frequent_pairs

def get_frequent_k_itemsets(file_path, frequent_itemsets_k_minus_1, k, support):
    # Generate candidate k-itemsets
    candidates = generate_candidates(frequent_itemsets_k_minus_1, k)
    candidate_counts = collections.defaultdict(int)
    
    # Count occurrences of candidates in the dataset
    with open(file_path, 'r') as f:
        for line in f:
            items = set(line.strip().split(' '))
            
            # Check each candidate against this basket
            for candidate in candidates:
                if set(candidate).issubset(items):
                    candidate_counts[candidate] += 1
    
    # Filter candidates that meet minimum support
    frequent_k_itemsets = {itemset: count for itemset, count in candidate_counts.items() 
                          if count >= support}
    return frequent_k_itemsets

def generate_candidates(frequent_itemsets_k_minus_1, k):
    candidates = []
    
    # k = 1: generate pairs from singletons
    if isinstance(frequent_itemsets_k_minus_1, set):
        return list(combinations(sorted(frequent_itemsets_k_minus_1), k))
    
    # k >= 2: use Apriori candidate generation
    # Convert to sorted list for efficient joining
    frequent_list = sorted([tuple(sorted(itemset)) for itemset in frequent_itemsets_k_minus_1.keys()])
    
    # More Efficiently: We have itemsets of size k-1, we want k itemset candidate, 
    # only if first k-2 items are the same should we combine
    # A simple example is we have (1,2,3) and (1,2,4), (3,4,5) as frequent itemsets of size 3, 
    # we can combine them to (1,2,3,4) but not (1,2,3,5)
    # because we know (1,2,5) is not frequent
    # This is a prune step in the apriori algorithm
    for i in range(len(frequent_list)):
        for j in range(i + 1, len(frequent_list)):
            itemset1 = frequent_list[i]
            itemset2 = frequent_list[j]

            # Check if first k-2 elements are the same
            if itemset1[:-1] == itemset2[:-1]:
                new_candidate = tuple(sorted(set(itemset1) | set(itemset2)))
                
                # Prune: check if all (k-1)-subsets are frequent
                subsets = combinations(new_candidate, k - 1)
                if all(tuple(sorted(subset)) in frequent_list for subset in subsets):
                    candidates.append(new_candidate)
    
    return candidates


In [26]:
def find_all_frequent_itemsets(file_path, s_percent):
    all_frequent_itemsets = {}
    
    print("Pass 1: Finding frequent singletons...")
    frequent_singletons, support, total_baskets, item_counts = load_and_count_singletons(file_path, s_percent)
    all_frequent_itemsets[1] = {(item,): item_counts[item] for item in frequent_singletons}  # 转换为统一格式
    print(f"Found {len(frequent_singletons)} frequent singletons")
    
    print("\nPass 2: Finding frequent pairs...")
    frequent_pairs = get_frequent_pairs(file_path, frequent_singletons, support)
    if not frequent_pairs:
        print("No frequent pairs found. Stopping.")
        return all_frequent_itemsets, support, total_baskets
    all_frequent_itemsets[2] = frequent_pairs
    print(f"Found {len(frequent_pairs)} frequent pairs")
    
    k = 3
    previous_frequent = frequent_pairs
    
    while previous_frequent:
        print(f"\nPass {k}: Finding frequent {k}-itemsets...")
        frequent_k_itemsets = get_frequent_k_itemsets(file_path, previous_frequent, k, support)
        
        if not frequent_k_itemsets:
            print(f"No frequent {k}-itemsets found. Stopping.")
            break
            
        all_frequent_itemsets[k] = frequent_k_itemsets
        print(f"Found {len(frequent_k_itemsets)} frequent {k}-itemsets")
        
        previous_frequent = frequent_k_itemsets
        k += 1
    
    return all_frequent_itemsets, support, total_baskets

filepath = "T10I4D100K.dat"
s_percent = 0.01

print(f"Mining frequent itemsets with minimum support = {s_percent * 100}%")
print("=" * 60)

all_frequent, support, total_baskets = find_all_frequent_itemsets(filepath, s_percent)

print("\n" + "=" * 60)
print("Summary of Frequent Itemsets:")
print("=" * 60)
total_count = 0
for k, itemsets in sorted(all_frequent.items()):
    count = len(itemsets)
    total_count += count
    print(f"{k}-itemsets: {count}")

print(f"\nTotal frequent itemsets: {total_count}")
print(f"Total baskets: {total_baskets}")
print(f"Minimum support count: {support}")

Mining frequent itemsets with minimum support = 1.0%
Pass 1: Finding frequent singletons...
Found 375 frequent singletons

Pass 2: Finding frequent pairs...
Found 9 frequent pairs

Pass 3: Finding frequent 3-itemsets...
Found 1 frequent 3-itemsets

Pass 4: Finding frequent 4-itemsets...
No frequent 4-itemsets found. Stopping.

Summary of Frequent Itemsets:
1-itemsets: 375
2-itemsets: 9
3-itemsets: 1

Total frequent itemsets: 385
Total baskets: 100000
Minimum support count: 1000


## Optional: Mining Assotiation Rule
If we have already found out all the frequent itemsets, how do we derive the association rule from them?

First of all, let us give a clear definition of the association rule by introducing the concept of confidence.

For an item set $I = \{i_1,i_2,i_3,...,i_k\}$ and $j$, the **if-then** rules states if a basket contains all of $i_1,i_2,i_3,...,i_k$, then it is **likely** to contain $j$

We model the likelihood by confidence, defined as 
$$ \text{conf}(I\rightarrow j) = \frac{\text{support}(I \cup j)}{\text{support}(I)}$$

This definition is directly analogous to the formula for conditional probability in classical probability theory:

$$ P(j|I) = \frac{P(j,I)}{P(I)}$$

To conclulde, we are looking for  $I\rightarrow j$ with reasonably high support and confidence. A rule of thumb is that support should be at least 0.01 and confidence should be at least 50%.

However, in practice there are many rules and we only want to find significant ones. Therefore, we define **interest** as
$$ \text{Interest}(I\rightarrow j) = \text{conf}(I \rightarrow j) - Pr[j] $$
to measure a rule is interesting. An intuitive meaning of interest is that if the customer's intention to buy $j$ is increased/decreased significantly if he has bought all item in $I$. Interesting rules are those with high positive or negative 
interest values (usually above 0.5)





Now, let's look into the algorithm of mining assocaition rules. 

To begin with, there is 2 approach. The first one is a naive solution, in which we generate all possible rules and examine their confidence. However, in the second approach, we can prune early according to the observation of monotocity. For example, if we have {m,c,b},{b,m},{c,m},{c,b},{c,j}. To mine association rule from {m,c,b}, there are at most 6 rules. We start with rules that has single consequence, namely (m,c->b), (m,b->c), (c,b->m). If we were to find out that (b,c->m) does not meet minimum confidence, we do not need to check (b->c,m) or (c->b,m), because 
$$minconf > \frac{s(m,c,b)}{s(b,m)} >= \frac{s(m,c,b)}{s(c)} , \frac{s(m,c,b)}{s(b)} $$

The pseudo code is listed below, and in our implementation we choose the naive one. For confidence, we choose 50% as minimal.

```
FOR EACH subset A of I
    // Since I is frequent, so is A

    Variant 1: Naive Solution
        B = I|A
        Rule = A -> B
        conf(A->B)=support(I)/support(A)
        
        if conf(A->B) >= min_conf:
            R.ADD(Rule)
    
    Variant 2: Prune by monotocity
        // Generate b such that  Rule I\b -> b
        H_k = { {b} | b in I }
        
        k = 1
        WHILE H_k IS NOT EMPTY
            H_k_next = {}

            FOR EACH B IN H_k // B is consequence set
                A = I \ B
                Rule = A -> B

                conf_AB = support(I)/support(A)
                
                // Prune
                IF conf_AB >= min_conf
                    R.ADD(RULE)
                    H_k_next.ADD(B)
                ELSE
                    Continue
            H_k - GENERATE_NEXT_CONSEQUENCE(H_k_next)
            k = k + 1
        
```


In [27]:
def mine_association_rules(all_frequent_itemsets, min_confidence=0.5, total_baskets=None):
    rules = []
    
    # Flatten frequent itemsets
    support_dict = {}
    for k, itemsets in all_frequent_itemsets.items():
        for itemset, count in itemsets.items():
            support_dict[itemset] = count
    
    # Generate rules
    for k in range(2, max(all_frequent_itemsets.keys()) + 1):
        if k not in all_frequent_itemsets:
            continue
            
        for itemset, itemset_support in all_frequent_itemsets[k].items():
            # Generate all non-empty proper subsets of the itemset as antecedents
            for i in range(1, len(itemset)):
                # Generate all subsets of size i
                for antecedent in combinations(itemset, i):
                    antecedent = tuple(sorted(antecedent))
                    consequent = tuple(sorted(set(itemset) - set(antecedent)))
                    
                    # Calculate confidence: conf(A->B) = support(A∪B) / support(A)
                    antecedent_support = support_dict.get(antecedent, 0)
                    
                    if antecedent_support > 0:
                        confidence = itemset_support / antecedent_support
                        
                        # If confidence meets the minimum threshold, add the rule
                        if confidence >= min_confidence:
                            rule = {
                                'antecedent': antecedent,
                                'consequent': consequent,
                                'support': itemset_support,
                                'confidence': confidence
                            }
                            
                            # Calculate interest (if total baskets provided)
                            if total_baskets:
                                consequent_support = support_dict.get(consequent, 0)
                                prob_consequent = consequent_support / total_baskets
                                rule['interest'] = confidence - prob_consequent
                            
                            rules.append(rule)
    
    return rules

# 使用示例
print("\n" + "=" * 60)
print("Mining Association Rules...")
print("=" * 60)

# 挖掘关联规则
rules = mine_association_rules(all_frequent, min_confidence=0.5, total_baskets=total_baskets)

print(f"\nFound {len(rules)} association rules with confidence >= 0.5")

# 显示前 10 条规则
print("\nTop 10 Rules by Confidence:")
print("-" * 60)
sorted_rules = sorted(rules, key=lambda x: x['confidence'], reverse=True)

for i, rule in enumerate(sorted_rules[:10], 1):
    ant = ', '.join(rule['antecedent'])
    cons = ', '.join(rule['consequent'])
    print(f"{i}. {ant} -> {cons}")
    print(f"   Support: {rule['support']}, Confidence: {rule['confidence']:.4f}", end='')
    if 'interest' in rule:
        print(f", Interest: {rule['interest']:.4f}")
    else:
        print()
    print()


Mining Association Rules...

Found 7 association rules with confidence >= 0.5

Top 10 Rules by Confidence:
------------------------------------------------------------
1. 704, 825 -> 39
   Support: 1035, Confidence: 0.9392, Interest: 0.8966

2. 39, 704 -> 825
   Support: 1035, Confidence: 0.9350, Interest: 0.9041

3. 39, 825 -> 704
   Support: 1035, Confidence: 0.8719, Interest: 0.8540

4. 704 -> 39
   Support: 1107, Confidence: 0.6171, Interest: 0.5745

5. 704 -> 825
   Support: 1102, Confidence: 0.6143, Interest: 0.5834

6. 227 -> 390
   Support: 1049, Confidence: 0.5770, Interest: 0.5502

7. 704 -> 39, 825
   Support: 1035, Confidence: 0.5769, Interest: 0.5651



## Misc
To state that our result is correct, we introduce the python library `mlxtend` to mine the frequent itemsets and association rules. The result is identical to our implementation.

In [28]:
# uv pip install mlxtend

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# 读取数据
transactions = []
with open('T10I4D100K.dat', 'r') as f:
    for line in f:
        items = line.strip().split(' ')
        transactions.append(items)

# 转换为 one-hot 编码的 DataFrame
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

# 找频繁项集
frequent_itemsets = apriori(df, min_support=0.01, use_colnames=True)
print(frequent_itemsets)

# 生成关联规则
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print(rules)

     support        itemsets
0    0.01535             (1)
1    0.01351            (10)
2    0.01749           (100)
3    0.01158           (104)
4    0.01100           (105)
..       ...             ...
380  0.01187       (39, 825)
381  0.01042      (390, 722)
382  0.01102      (704, 825)
383  0.01194      (789, 829)
384  0.01035  (39, 825, 704)

[385 rows x 2 columns]
  antecedents consequents  antecedent support  consequent support  support  \
0       (227)       (390)             0.01818             0.02685  0.01049   
1       (704)        (39)             0.01794             0.04258  0.01107   
2       (704)       (825)             0.01794             0.03085  0.01102   
3   (39, 825)       (704)             0.01187             0.01794  0.01035   
4   (39, 704)       (825)             0.01107             0.03085  0.01035   
5  (704, 825)        (39)             0.01102             0.04258  0.01035   
6       (704)   (39, 825)             0.01794             0.01187  0.01035   

   

In [29]:

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("Frequent Itemsets Count:")
print(f"Singletons (1-itemsets): {len(frequent_itemsets[frequent_itemsets['length'] == 1])}")
print(f"Doubletons (2-itemsets): {len(frequent_itemsets[frequent_itemsets['length'] == 2])}")
print(f"Tripletons (3-itemsets): {len(frequent_itemsets[frequent_itemsets['length'] == 3])}")
    
print("\nFrequent Itemsets Grouped by Size:")
print(frequent_itemsets.groupby('length').size())
    
print("\nAll Frequent Itemsets:")
print(frequent_itemsets)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print(f"\nNumber of Association Rules: {len(rules)}")
print(rules)

Frequent Itemsets Count:
Singletons (1-itemsets): 375
Doubletons (2-itemsets): 9
Tripletons (3-itemsets): 1

Frequent Itemsets Grouped by Size:
length
1    375
2      9
3      1
dtype: int64

All Frequent Itemsets:
     support        itemsets  length
0    0.01535             (1)       1
1    0.01351            (10)       1
2    0.01749           (100)       1
3    0.01158           (104)       1
4    0.01100           (105)       1
..       ...             ...     ...
380  0.01187       (39, 825)       2
381  0.01042      (390, 722)       2
382  0.01102      (704, 825)       2
383  0.01194      (789, 829)       2
384  0.01035  (39, 825, 704)       3

[385 rows x 3 columns]

Number of Association Rules: 7
  antecedents consequents  antecedent support  consequent support  support  \
0       (227)       (390)             0.01818             0.02685  0.01049   
1       (704)        (39)             0.01794             0.04258  0.01107   
2       (704)       (825)             0.01794      

In [30]:
print("=" * 80)
print("All Association Rules (using mlxtend)")
print("=" * 80)

# Display total number of rules
print(f"\nTotal number of rules: {len(rules)}")

# Display basic statistics of the rules
print("\nRules Statistics:")
print(f"Average confidence: {rules['confidence'].mean():.4f}")
print(f"Max confidence: {rules['confidence'].max():.4f}")
print(f"Min confidence: {rules['confidence'].min():.4f}")
print(f"Average lift: {rules['lift'].mean():.4f}")

# Display all rules sorted by confidence
print("\n" + "=" * 80)
print("All Rules (sorted by confidence):")
print("=" * 80)

sorted_rules = rules.sort_values('confidence', ascending=False)

for idx, row in sorted_rules.iterrows():
    antecedents = ', '.join(list(row['antecedents']))
    consequents = ', '.join(list(row['consequents']))
    
    print(f"\nRule {idx + 1}:")
    print(f"  {antecedents} -> {consequents}")
    print(f"  Support: {row['support']:.4f}")
    print(f"  Confidence: {row['confidence']:.4f}")
    print(f"  Lift: {row['lift']:.4f}")
    print(f"  Leverage: {row['leverage']:.4f}")
    print(f"  Conviction: {row['conviction']:.4f}")


All Association Rules (using mlxtend)

Total number of rules: 7

Rules Statistics:
Average confidence: 0.7331
Max confidence: 0.9392
Min confidence: 0.5769
Average lift: 29.3520

All Rules (sorted by confidence):

Rule 6:
  704, 825 -> 39
  Support: 0.0103
  Confidence: 0.9392
  Lift: 22.0573
  Leverage: 0.0099
  Conviction: 15.7474

Rule 5:
  39, 704 -> 825
  Support: 0.0103
  Confidence: 0.9350
  Lift: 30.3066
  Leverage: 0.0100
  Conviction: 14.9007

Rule 4:
  39, 825 -> 704
  Support: 0.0103
  Confidence: 0.8719
  Lift: 48.6035
  Leverage: 0.0101
  Conviction: 7.6691

Rule 2:
  704 -> 39
  Support: 0.0111
  Confidence: 0.6171
  Lift: 14.4917
  Leverage: 0.0103
  Conviction: 2.5002

Rule 3:
  704 -> 825
  Support: 0.0110
  Confidence: 0.6143
  Lift: 19.9115
  Leverage: 0.0105
  Conviction: 2.5125

Rule 1:
  227 -> 390
  Support: 0.0105
  Confidence: 0.5770
  Lift: 21.4900
  Leverage: 0.0100
  Conviction: 2.3006

Rule 7:
  704 -> 39, 825
  Support: 0.0103
  Confidence: 0.5769
  Lift: